# 05. Детектор деградації

**Етап 05 з 13** · NASA C-MAPSS FD001 · RUL prediction


## 1. Дані

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from xgboost import XGBRegressor

# Шлях до даних. На Colab: /content/drive/MyDrive/predictive_maintenance
DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))
SEED = 42

COLUMNS = ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

In [ ]:
def load_train(data_dir):
    """Читає train_FD001.txt і додає RUL: скільки циклів лишилось до останнього циклу двигуна."""
    train = pd.read_csv(data_dir / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)
    train["RUL"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]
    return train

train = load_train(DATA_DIR)

# Поділ за двигунами: 80 двигунів на train, 20 на validation
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["unit_id"]))

train_df = train.iloc[train_idx].copy()
val_df = train.iloc[val_idx].copy()

print("Train engines:", train_df["unit_id"].nunique(), "| rows:", len(train_df))
print("Validation engines:", val_df["unit_id"].nunique(), "| rows:", len(val_df))

## 2. Ознаки

Детектор: baseline двигуна це медіана перших `baseline_cycles` циклів, масштаби сенсорів і поріг
оцінюються тільки на train-двигунах. Перші `baseline_cycles` рядків кожного двигуна є калібруванням,
їхні ознаки детектора = NaN.

In [ ]:
class DegradationDetector:
    """Детектор стійких направлених відхилень сенсорів, причинні ознаки по кожному двигуну.

    Кроки: baseline двигуна з перших baseline_cycles циклів; глобальні масштаби й поріг
    з train-двигунів; directional score; trailing rolling mean; підтвердження після
    persistence послідовних перевищень порога. Ознаки з'являються лише після підтвердження.
    """

    def __init__(
        self,
        baseline_cycles=20,
        window=10,
        persistence=5,
        threshold_quantile=0.99,
        min_scale=1e-6,
    ):
        if baseline_cycles < 1:
            raise ValueError("baseline_cycles must be >= 1")

        if window < 1:
            raise ValueError("window must be >= 1")

        if persistence < 1:
            raise ValueError("persistence must be >= 1")

        if not 0 < threshold_quantile < 1:
            raise ValueError(
                "threshold_quantile must be between 0 and 1"
            )

        self.baseline_cycles = baseline_cycles
        self.window = window
        self.persistence = persistence
        self.threshold_quantile = threshold_quantile
        self.min_scale = min_scale

        # Очікуваний напрям сенсора при деградації:
        # +1 зростання дає внесок у score, -1 спадання дає внесок у score.
        self.directions = {
            "sensor_4": 1,
            "sensor_7": -1,
            "sensor_11": 1,
            "sensor_12": -1,
            "sensor_15": 1,
            "sensor_20": -1,
            "sensor_21": -1,
        }

        self.sensors = list(self.directions)

        self.scale_ = None
        self.threshold_ = None
        self.is_fitted_ = False

    def _validate_columns(self, df):
        """Перевіряє, що в df є unit_id, cycle і всі сенсори детектора."""
        required = {
            "unit_id",
            "cycle",
            *self.sensors,
        }

        missing = required - set(df.columns)

        if missing:
            raise ValueError(
                f"Missing required columns: {sorted(missing)}"
            )

    def _calculate_scores(self, engine):
        """Рахує відхилення сенсорів від baseline двигуна, directional score і його згладжену версію.

        Повертає: відсортовані дані двигуна, стандартизовані відхилення, raw_score, smooth_score.
        """

        engine = engine.sort_values("cycle").copy()

        # Baseline двигуна: медіана перших B спостережень.
        baseline = (
            engine[self.sensors]
            .iloc[:self.baseline_cycles]
            .median()
        )

        # Відхилення кожного сенсора від його baseline у масштабах train.
        deviation = (
            engine[self.sensors]
            .sub(baseline, axis="columns")
            .div(self.scale_, axis="columns")
        )

        # Залишаємо відхилення в очікуваному напрямі деградації.
        direction = pd.Series(self.directions)

        directional_deviation = (
            deviation
            .mul(direction, axis="columns")
            .clip(lower=0)
        )

        # Середнє по сенсорах.
        raw_score = directional_deviation.mean(axis=1)

        # Згладжування лише поточним і минулими значеннями.
        smooth_score = (
            raw_score
            .rolling(
                window=self.window,
                min_periods=1,
            )
            .mean()
        )

        return engine, deviation, raw_score, smooth_score

    def fit(self, train_df):
        """Оцінює глобальні масштаби сенсорів і поріг. Викликати лише з train-двигунами."""

        self._validate_columns(train_df)

        calibration_residuals = []

        # залишки всередині калібрувального вікна кожного двигуна.
        for _, group in train_df.groupby("unit_id", sort=False):
            engine = group.sort_values("cycle")

            # Двигуни з коротшою історією не дають повного калібрувального вікна.
            if len(engine) < self.baseline_cycles:
                continue

            calibration = engine.iloc[:self.baseline_cycles]

            baseline = calibration[self.sensors].median()

            residuals = (
                calibration[self.sensors]
                .sub(baseline, axis="columns")
            )

            calibration_residuals.append(residuals)

        if not calibration_residuals:
            raise ValueError(
                "No engine has enough observations for calibration"
            )

        residuals_all = pd.concat(
            calibration_residuals,
            ignore_index=True,
        )

        # масштаб кожного сенсора (MAD * 1.4826).
        self.scale_ = (
            residuals_all
            .abs()
            .median()
            .mul(1.4826)
            .clip(lower=self.min_scale)
            .reindex(self.sensors)
        )

        #score у калібрувальних вікнах.
        calibration_scores = []

        for _, group in train_df.groupby("unit_id", sort=False):
            engine = group.sort_values("cycle")

            if len(engine) < self.baseline_cycles:
                continue

            _, _, _, smooth_score = self._calculate_scores(engine)

            scores = smooth_score.iloc[:self.baseline_cycles]

            calibration_scores.extend(scores.to_numpy())

        calibration_scores = np.asarray(
            calibration_scores,
            dtype=float,
        )

        calibration_scores = calibration_scores[
            np.isfinite(calibration_scores)
        ]

        if len(calibration_scores) == 0:
            raise ValueError(
                "Could not calculate valid calibration scores"
            )

        # поріг як квантиль калібрувальних score.
        self.threshold_ = float(
            np.quantile(
                calibration_scores,
                self.threshold_quantile,
            )
        )

        self.n_calibration_scores_ = len(calibration_scores)

        self.is_fitted_ = True

        return self

    def transform(self, df):
        """Рахує причинні ознаки детектора для кожного двигуна. Рядки до підтвердження не заповнюються назад."""

        if not self.is_fitted_:
            raise RuntimeError(
                "Call fit(train_df) before transform(df)"
            )

        self._validate_columns(df)

        results = []

        for _, group in df.groupby("unit_id", sort=False):
            engine, deviation, _, smooth_score = (
                self._calculate_scores(group)
            )

            n = len(engine)

            cycles = engine["cycle"].to_numpy(dtype=float)
            scores = smooth_score.to_numpy(dtype=float)
            deviations = deviation.to_numpy(dtype=float)

            baseline_ready = np.zeros(n, dtype=np.int8)
            baseline_ready[self.baseline_cycles:] = 1

            degradation_detected = np.zeros(n, dtype=np.int8)

            # Числові ознаки недоступні під час калібрування.
            degradation_score = np.full(n, np.nan)
            threshold_excess = np.full(n, np.nan)

            degradation_detection_cycle = np.full(n, np.nan)
            degradation_start_estimate = np.full(n, np.nan)

            cycles_since_detection = np.full(n, np.nan)
            cycles_since_degradation = np.full(n, np.nan)

            trend_since_degradation = np.full(n, np.nan)

            sensor_delta = {
                sensor: np.full(n, np.nan)
                for sensor in self.sensors
            }

            # Score доступний тільки після калібрування.
            for i in range(self.baseline_cycles, n):
                degradation_score[i] = scores[i]

                threshold_excess[i] = max(
                    0.0,
                    scores[i] - self.threshold_,
                )

            # Підтвердження: persistence послідовних перевищень порога.
            consecutive = 0
            detected = False

            detection_idx = None
            onset_idx = None

            for i in range(self.baseline_cycles, n):

                if not detected:
                    if scores[i] > self.threshold_:
                        consecutive += 1
                    else:
                        consecutive = 0

                    if consecutive >= self.persistence:
                        detected = True

                        # Поточний цикл — момент підтвердження.
                        detection_idx = i

                        # Оцінка початку серії, доступна лише зараз.
                        onset_idx = i - self.persistence + 1

                # Ознаки з'являються з моменту підтвердження і не заповнюються назад.
                if detected:
                    degradation_detected[i] = 1

                    degradation_detection_cycle[i] = (
                        cycles[detection_idx]
                    )

                    degradation_start_estimate[i] = (
                        cycles[onset_idx]
                    )

                    cycles_since_detection[i] = (
                        cycles[i] - cycles[detection_idx]
                    )

                    cycles_since_degradation[i] = (
                        cycles[i] - cycles[onset_idx]
                    )

                    # Нахил згладженого score від оціненого початку до поточного циклу.
                    history_cycles = cycles[onset_idx:i + 1]
                    history_scores = scores[onset_idx:i + 1]

                    if (
                        len(history_scores) >= 2
                        and np.ptp(history_cycles) > 0
                    ):
                        trend_since_degradation[i] = np.polyfit(
                            history_cycles,
                            history_scores,
                            1,
                        )[0]
                    else:
                        trend_since_degradation[i] = 0.0

                    # Зміна сенсорів відносно оціненого початку.
                    delta = (
                        deviations[i] - deviations[onset_idx]
                    )

                    for j, sensor in enumerate(self.sensors):
                        sensor_delta[sensor][i] = delta[j]

            features = pd.DataFrame(
                {
                    "baseline_ready": baseline_ready,
                    "degradation_detected": degradation_detected,
                    "degradation_score": degradation_score,
                    "threshold_excess": threshold_excess,
                    "degradation_detection_cycle": (
                        degradation_detection_cycle
                    ),
                    "degradation_start_estimate": (
                        degradation_start_estimate
                    ),
                    "cycles_since_detection": cycles_since_detection,
                    "cycles_since_degradation": (
                        cycles_since_degradation
                    ),
                    "trend_since_degradation": (
                        trend_since_degradation
                    ),
                },
                index=engine.index,
            )

            for sensor in self.sensors:
                features[
                    f"{sensor}_dev_since_estimated_onset"
                ] = sensor_delta[sensor]

            results.append(features)

        # Повертаємо початковий порядок і індекс рядків.
        return pd.concat(results).reindex(df.index)

In [ ]:
# Параметри детектора, однакові для validation і CV
DETECTOR_PARAMS = dict(baseline_cycles=20, window=10, persistence=5, threshold_quantile=0.99)

# Масштаби і поріг навчаються лише на train.
detector = DegradationDetector(**DETECTOR_PARAMS)
detector.fit(train_df)

train_detector_features = detector.transform(train_df)
val_detector_features = detector.transform(val_df)

print(f"Threshold: {detector.threshold_:.6f}")
print(f"Calibration observations: {detector.n_calibration_scores_}")
print("Train features:", train_detector_features.shape, "| validation features:", val_detector_features.shape)

### Діагностика порогу і підтвердження

In [ ]:
diagnostics = (
    pd.concat(
        [train_df[["unit_id", "cycle"]], train_detector_features],
        axis=1,
    )
    .query("degradation_detected == 1")
    .groupby("unit_id")
    .agg(
        first_detection=("degradation_start_estimate", "first"),
        max_degradation_age=("cycles_since_degradation", "max"),
    )
)

print("Detected engines:", len(diagnostics), "of", train_df["unit_id"].nunique())
display(diagnostics.describe())

In [ ]:
unit_id = train_df["unit_id"].iloc[0]

engine = train_df[train_df["unit_id"] == unit_id].sort_values("cycle")
features = train_detector_features.loc[engine.index]

plt.figure(figsize=(13, 5))

plt.plot(
    engine["cycle"].to_numpy(),
    features["degradation_score"].to_numpy(),
    label="Degradation score",
)

plt.axhline(
    y=detector.threshold_,
    linestyle="--",
    label=f"Threshold = {detector.threshold_:.3f}",
)

detected = features["degradation_detected"].eq(1)

if detected.any():
    onset_cycle = features.loc[detected, "degradation_start_estimate"].iloc[0]

    plt.axvline(
        x=onset_cycle,
        linestyle=":",
        label=f"Estimated onset at cycle {onset_cycle:.0f}",
    )

plt.title(f"Engine {unit_id}: Degradation Detection")
plt.xlabel("Cycle")
plt.ylabel("Degradation score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Двигун 2: перевищення порога після калібрування і довжина серій перевищень.
ENGINE_ID = 2

engine_raw = (
    pd.concat([train_df, val_df], axis=0)
    .loc[lambda d: d["unit_id"] == ENGINE_ID]
    .sort_values("cycle")
)

engine_debug = pd.concat(
    [
        engine_raw[["unit_id", "cycle"]],
        detector.transform(engine_raw),
    ],
    axis=1,
)

display(
    engine_debug.loc[
        engine_debug["cycle"].between(120, 150),
        ["cycle", "baseline_ready", "degradation_score", "degradation_detected", "degradation_start_estimate"],
    ]
)

above = (
    engine_debug["baseline_ready"].eq(1)
    & engine_debug["degradation_score"].gt(detector.threshold_)
)

# Новий run_id щоразу, коли стан перевищення змінюється.
run_id = above.ne(above.shift(fill_value=False)).cumsum()

runs = (
    engine_debug.loc[above]
    .groupby(run_id[above])
    .agg(
        start_cycle=("cycle", "min"),
        end_cycle=("cycle", "max"),
        consecutive_cycles=("cycle", "size"),
        max_score=("degradation_score", "max"),
    )
)

display(runs)

In [ ]:
# Серія експериментів з persistence: перша серія довжиною >= P дає onset і підтвердження.
PERSISTENCE_VALUES = [5, 8, 10, 15]

persistence_rows = []

for p in PERSISTENCE_VALUES:
    eligible_runs = runs[runs["consecutive_cycles"] >= p]

    if eligible_runs.empty:
        persistence_rows.append({"persistence": p, "onset_cycle": np.nan, "confirmation_cycle": np.nan})
        continue

    start_cycle = int(eligible_runs.iloc[0]["start_cycle"])

    persistence_rows.append({
        "persistence": p,
        "onset_cycle": start_cycle,
        "confirmation_cycle": start_cycle + p - 1,
    })

persistence_df = pd.DataFrame(persistence_rows)
display(persistence_df)

### Каталог ознак детектора (групи A–E)

- **A. detector_state**: `baseline_ready`, `degradation_detected`. Чи завершено калібрування, чи підтверджено стійке відхилення.
- **B. degradation_score**: `degradation_score`, `threshold_excess`. Згладжений score і його перевищення порога.
- **C. degradation_timing**: `degradation_detection_cycle`, `degradation_start_estimate`, `cycles_since_detection`, `cycles_since_degradation`. Часові характеристики від підтвердження і оціненого початку.
- **D. degradation_dynamics**: `trend_since_degradation`. Нахил score від оціненого початку до поточного циклу.
- **E. sensor_deviation**: `sensor_{4,7,11,12,15,20,21}_dev_since_estimated_onset`. Зміна окремих сенсорів від оціненого початку.

In [ ]:
DEGRADATION_FEATURE_GROUPS = {
    "detector_state": [
        "baseline_ready",
        "degradation_detected",
    ],
    "degradation_score": [
        "degradation_score",
        "threshold_excess",
    ],
    "degradation_timing": [
        "degradation_detection_cycle",
        "degradation_start_estimate",
        "cycles_since_detection",
        "cycles_since_degradation",
    ],
    "degradation_dynamics": [
        "trend_since_degradation",
    ],
    "sensor_deviation": [
        "sensor_4_dev_since_estimated_onset",
        "sensor_7_dev_since_estimated_onset",
        "sensor_11_dev_since_estimated_onset",
        "sensor_12_dev_since_estimated_onset",
        "sensor_15_dev_since_estimated_onset",
        "sensor_20_dev_since_estimated_onset",
        "sensor_21_dev_since_estimated_onset",
    ],
}

DEGRADATION_FEATURES = [
    feature
    for group in DEGRADATION_FEATURE_GROUPS.values()
    for feature in group
]

for group_name, features in DEGRADATION_FEATURE_GROUPS.items():
    print(f"{group_name}: {len(features)} features")

print("Total degradation features:", len(DEGRADATION_FEATURES))

### Базові ознаки і модель

In [ ]:
BASELINE_FEATURES = (
    ["cycle", "setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

XGB_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.012074507989018332,
    "max_depth": 4,
    "min_child_weight": 3.2298041995296414,
    "subsample": 0.7900204897773602,
    "colsample_bytree": 0.9223090698240903,
    "reg_alpha": 0.7459635164536705,
    "reg_lambda": 4.416722489003329,
    "gamma": 0.2695542352154229,
    "objective": "reg:squarederror",
    "random_state": SEED,
    "n_jobs": -1,
    "tree_method": "hist",
}

# Перевірка: усі ознаки детектора є, рядки вирівняні з train_df і val_df.
assert not set(DEGRADATION_FEATURES) - set(train_detector_features.columns)
assert not set(DEGRADATION_FEATURES) - set(val_detector_features.columns)
assert train_detector_features.index.equals(train_df.index)
assert val_detector_features.index.equals(val_df.index)

y_train = train_df["RUL"]
y_val = val_df["RUL"]

In [ ]:
def build_X(base_df, det_df, extra_features):
    """Повертає базові ознаки, до яких додано вибрані ознаки детектора."""
    base = base_df[BASELINE_FEATURES]

    if not extra_features:
        return base.copy()

    return pd.concat([base, det_df[extra_features]], axis=1)


def regression_metrics(y_true, y_pred):
    """Повертає MAE, RMSE і R2 для прогнозу, обмеженого нулем."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def evaluate_model(model, X_train, y_train, X_val, y_val):
    """Навчає модель і повертає метрики на validation. RUL не може бути від'ємним."""
    model.fit(X_train, y_train)
    predictions = np.maximum(model.predict(X_val), 0)
    return regression_metrics(y_val, predictions)

## 3. Експерименти

Усі конфігурації лежать в одному словнику `EXPERIMENTS`,
а запускаються одним циклом. Вхідний baseline завжди `BASELINE_FEATURES`.

Ablation («Without …») — це ALL без групи, групи «Baseline + …» — одна група. Назву «Score only» з
оригінальної клітинки з `ablation`-таблицею зі 2 ознаками перейменовано на «Baseline + degradation_score»
(це `degradation_score` + `threshold_excess`); одноознакову конфігурацію залишено як «Score only».

In [ ]:
G = DEGRADATION_FEATURE_GROUPS
SCORE_GROUP = G["degradation_score"]
OTHER_GROUPS = [g for g in G if g != "degradation_score"]

EXPERIMENTS = {"Baseline": []}

EXPERIMENTS["Baseline + All degradation"] = DEGRADATION_FEATURES

for g in G:
    EXPERIMENTS[f"Baseline + {g}"] = G[g]

for g in G:
    EXPERIMENTS[f"Without {g}"] = [f for f in DEGRADATION_FEATURES if f not in G[g]]

EXPERIMENTS["Score only"] = ["degradation_score"]
EXPERIMENTS["Threshold excess only"] = ["threshold_excess"]
EXPERIMENTS["Score + dynamics"] = ["degradation_score", "trend_since_degradation"]

for g in OTHER_GROUPS:
    EXPERIMENTS[f"Score group + {g}"] = SCORE_GROUP + G[g]

print("Configurations:", len(EXPERIMENTS))

val_rows = []

for name, extra in EXPERIMENTS.items():
    X_tr = build_X(train_df, train_detector_features, extra)
    X_va = build_X(val_df, val_detector_features, extra)

    metrics = evaluate_model(
        XGBRegressor(**XGB_PARAMS),
        X_tr,
        y_train,
        X_va,
        y_val,
    )

    val_rows.append({
        "Experiment": name,
        "N_features": X_tr.shape[1],
        **metrics,
    })

val_results = pd.DataFrame(val_rows)

### Перевірка на 5 фолдах (GroupKFold по train-двигунах)

In [ ]:
CV_CONFIGS = {
    "Baseline": [],
    "Baseline + score": ["degradation_score"],
    "Baseline + score + dynamics": ["degradation_score", "trend_since_degradation"],
}


def run_cv(configs, n_splits=5):
    """Для кожного фолду навчає новий детектор лише на train-частині; повертає метрики фолдів і OOF-підсумок."""
    assert "RUL" in train.columns and train["RUL"].notna().all()
    assert train["unit_id"].nunique() >= n_splits

    fold_rows = []
    oof = {name: pd.Series(index=train.index, dtype=float) for name in configs}

    for fold, (tr_idx, va_idx) in enumerate(
        GroupKFold(n_splits=n_splits).split(train, groups=train["unit_id"]),
        start=1,
    ):
        fold_train = train.iloc[tr_idx].copy()
        fold_val = train.iloc[va_idx].copy()

        # Детектор не бачить RUL; новий детектор на кожному фолді.
        train_raw = fold_train.drop(columns=["RUL"])
        val_raw = fold_val.drop(columns=["RUL"])

        detector_fold = DegradationDetector(**DETECTOR_PARAMS)
        detector_fold.fit(train_raw)

        tr_det = detector_fold.transform(train_raw)
        va_det = detector_fold.transform(val_raw)

        assert tr_det.index.equals(fold_train.index)
        assert va_det.index.equals(fold_val.index)

        for name, extra in configs.items():
            X_tr = build_X(fold_train, tr_det, extra)
            X_va = build_X(fold_val, va_det, extra)

            model = XGBRegressor(**XGB_PARAMS)
            model.fit(X_tr, fold_train["RUL"])

            predictions = np.maximum(model.predict(X_va), 0)

            oof[name].loc[fold_val.index] = predictions

            fold_rows.append({
                "Fold": fold,
                "Model": name,
                "N_features": X_tr.shape[1],
                "N_val_engines": fold_val["unit_id"].nunique(),
                **regression_metrics(fold_val["RUL"], predictions),
            })

        print(f"Fold {fold}/{n_splits} completed")

    fold_df = pd.DataFrame(fold_rows)

    cv_summary = (
        fold_df
        .groupby("Model")
        .agg(
            CV_MAE_mean=("MAE", "mean"),
            CV_MAE_std=("MAE", "std"),
            CV_RMSE_mean=("RMSE", "mean"),
            CV_RMSE_std=("RMSE", "std"),
            CV_R2_mean=("R2", "mean"),
            CV_R2_std=("R2", "std"),
        )
        .reset_index()
    )

    oof_rows = []
    for name, predictions in oof.items():
        assert predictions.notna().all()
        oof_rows.append({"Model": name, **{
            f"OOF_{k}": v for k, v in regression_metrics(train["RUL"], predictions).items()
        }})

    cv_summary = cv_summary.merge(pd.DataFrame(oof_rows), on="Model")

    return fold_df, cv_summary.sort_values("OOF_MAE")


cv_folds, cv_summary = run_cv(CV_CONFIGS)
display(cv_folds.round(4))
display(cv_summary.round(4))

## 4. Результати

In [ ]:
summary = val_results.copy()

baseline_mae = summary.loc[summary["Experiment"] == "Baseline", "MAE"].iloc[0]
summary["MAE change vs baseline"] = summary["MAE"] - baseline_mae

summary = summary.sort_values("MAE").reset_index(drop=True)
display(summary.round(4))

## Висновки

**Validation (80/20 двигунів, XGBoost, 25 базових ознак):**
- Baseline: MAE 23.0717, RMSE 29.8586, R2 0.7932.
- Baseline + всі 16 ознак детектора (41 ознака): MAE 18.9782 (−4.09).
- Одна група до baseline: `degradation_score` дає −4.10 (18.9746), `degradation_dynamics` −3.30 (19.7672),
  `degradation_timing` −2.94 (20.1358), `sensor_deviation` −2.10 (20.9716), `detector_state` −1.70 (21.3723).
- Одна ознака `degradation_score` дає 18.9638, `threshold_excess` окремо 19.0164.
- Найкраща конфігурація на validation: `Score + dynamics` (`degradation_score` + `trend_since_degradation`): MAE 18.7226.
  Додавання `threshold_excess` до неї дає 18.7692.
- Ablation (ALL без групи): без `degradation_timing` MAE 18.8070 і без `degradation_score` 18.8133 є кращими за повну
  модель; без `degradation_dynamics` 19.2873 (гірше на 0.31). Без `detector_state` і без `sensor_deviation` різниця ≤ 0.002.

**5-фолдова GroupKFold по 100 train-двигунах:**
- Baseline: CV MAE 26.1027 ± 1.7422, OOF MAE 26.1019.
- Baseline + score: CV MAE 21.3006 ± 1.3315, OOF MAE 21.2997.
- Baseline + score + dynamics: CV MAE 21.2573 ± 1.3187, OOF MAE 21.2564.

Висновок: ознаки детектора (найбільше `degradation_score`) стабільно знижують MAE приблизно на 4–5 з 26 на CV.
Додаткова користь `trend_since_degradation` (≈ 0.04 на CV, ≈ 0.2 на validation) менша за розкид між фолдами (std ≈ 1.3),
тому її вклад потребує окремої перевірки. Вибір конфігурацій зроблено за validation, CV є незалежною перевіркою.